# Data Cleaning e Pré-processamento - FIAPMobile

**Objetivo:** Realizar a limpeza inicial, correção de tipagem, tratamento de valores ausentes e filtragem de colunas (Feature Selection inicial) para garantir a integridade da base de dados.

## Configuração do Ambiente

In [ ]:
# Importanto libs

# Operacionalização dados
import pandas as pd
import numpy as np

# Sistema operacional
import os

# Aumentando numero de colunas visualizaveis
pd.set_option('display.max_columns', None)

## Carregando e Descrição dos dados

### Sobre o dataset Telco Customer Churn

**Fonte**: Conjunto de dados de inventário de clientes da IBM ([disponibilizado via Kaggle/UCI](https://www.kaggle.com/datasets/yeanzc/telco-customer-churn-ibm-dataset)) contendo informações sobre uma empresa de telecomunicações que forneceu serviços de telefonia residencial e internet para 7043 clientes na Califórnia no terceiro trimestre.
* **Observações**: 7043
* **Coluna**: 33
* **Target**: `Churn Value`: (0 = Não saiu; 1 = Saiu) ou `Churn Label`: (No/Yes)
---

### Esquema das Variáveis (Dicionário de Dados)

#### Informações de Identificação e Localização
* **CustomerID**: ID exclusivo que identifica cada cliente.
* **Count**: Valor usado em relatórios para somar o número de clientes.
* **Country / State / City**: Localização de residência principal do cliente (País, Estado e Cidade).
* **Zip Code**: Código postal da residência.
* **Lat Long / Latitude / Longitude**: Dados geográficos de localização do cliente.

#### Perfil do Cliente (Demográfico)
* **Gender**: Gênero do cliente (Male, Female).
* **Senior Citizen**: Indica se o cliente tem 65 anos ou mais (Yes, No).
* **Partner**: Indica se o cliente tem um parceiro/companheiro (Yes, No).
* **Dependents**: Indica se o cliente vive com dependentes (filhos, pais, etc.) (Yes, No).

#### Serviços Contratados
* **Tenure Months**: Total de meses que o cliente está na empresa.
* **Phone Service**: Indica se o cliente possui serviço de telefonia fixa (Yes, No).
* **Multiple Lines**: Indica se o cliente possui várias linhas telefônicas (Yes, No).
* **Internet Service**: Tipo de serviço de internet (No, DSL, Fiber Optic, Cable).
* **Serviços Adicionais**: Online Security, Online Backup, Device Protection, Tech Support (Serviços extras de segurança, backup, proteção e suporte técnico) (Yes, No).
* **Streaming TV / Movies**: Indica se o cliente utiliza a internet para streaming de terceiros (Yes, No).

#### Informações Contratuais e Financeiras
* **Contract**: Tipo de contrato atual (Month-to-Month, One Year, Two Year).
* **Paperless Billing**: Indica se o cliente optou por fatura digital (Yes, No).
* **Payment Method**: Forma de pagamento (Bank Withdrawal, Credit Card, Mailed Check).
* **Monthly Charge**: Valor total da cobrança mensal atual.
* **Total Charges**: Cobranças totais acumuladas até o final do trimestre.

#### Métricas de Churn (Alvo)
* **Churn Label**: Rótulo qualitativo (Yes = saiu; No = permaneceu).
* **Churn Value**: Valor quantitativo para modelagem (1 = saiu; 0 = permaneceu).
* **Churn Score**: Valor de 0 a 100 calculado pelo IBM SPSS indicando a probabilidade de saída.
* **CLTV**: Customer Lifetime Value (Valor do tempo de vida do cliente).
* **Churn Reason**: Motivo específico pelo qual o cliente deixou a empresa.

In [3]:
archive_path = os.path.join('..', 'data', 'raw', 'Telco_customer_churn.xlsx')

try:
    df = pd.read_excel(archive_path)
    print("Arquivo carregado com sucesso!")
    print(f"Dimensão do dataset:\nlinhas: {df.shape[0]}\nColunas: {df.shape[1]}")
except FileNotFoundError:
    print("Erro: Arquivo não foi encontrado.")
except Exception as e:
    print(f"Erro: {e}")
    
df.head(10)

Arquivo carregado com sucesso!
Dimensão do dataset:
linhas: 7043
Colunas: 33


,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,Yes,Yes,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,No,Yes,49,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices
5,4190-MFLUW,1,United States,California,Los Angeles,90020,"34.066367, -118.309868",34.066367,-118.309868,Female,No,Yes,No,10,Yes,No,DSL,No,No,Yes,Yes,No,No,Month-to-month,No,Credit card (automatic),55.20,528.35,Yes,1,78,5925,Competitor offered higher download speeds
6,8779-QRDMV,1,United States,California,Los Angeles,90022,"34.02381, -118.156582",34.023810,-118.156582,Male,Yes,No,No,1,No,No phone service,DSL,No,No,Yes,No,No,Yes,Month-to-month,Yes,Electronic check,39.65,39.65,Yes,1,100,5433,Competitor offered more data
7,1066-JKSGK,1,United States,California,Los Angeles,90024,"34.066303, -118.435479",34.066303,-118.435479,Male,No,No,No,1,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,20.15,20.15,Yes,1,92,4832,Competitor made better offer
8,6467-CHFZW,1,United States,California,Los Angeles,90028,"34.099869, -118.326843",34.099869,-118.326843,Male,No,Yes,Yes,47,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.35,4749.15,Yes,1,77,5789,Competitor had better devices
9,8665-UTDHZ,1,United States,California,Los Angeles,90029,"34.089953, -118.294824",34.089953,-118.294824,Male,No,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,No,Electronic check,30.20,30.2,Yes,1,97,2915,Competitor had better devices


### Informações gerais e estatística descritiva do dataset

In [4]:
df.info()
print('*='*20)
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   str    
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   str    
 3   State              7043 non-null   str    
 4   City               7043 non-null   str    
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   str    
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   str    
 10  Senior Citizen     7043 non-null   str    
 11  Partner            7043 non-null   str    
 12  Dependents         7043 non-null   str    
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   str    
 15  Multiple Lines     7043 non-null   str    
 16  Internet Service   7043 non-null   

,Count,Zip Code,Latitude,Longitude,Tenure Months,Monthly Charges,Churn Value,Churn Score,CLTV
count,7043.0,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000
mean,1.0,93521.964646,36.282441,-119.798880,32.371149,64.761692,0.265370,58.699418,4400.295755
std,0.0,1865.794555,2.455723,2.157889,24.559481,30.090047,0.441561,21.525131,1183.057152
min,1.0,90001.000000,32.555828,-124.301372,0.000000,18.250000,0.000000,5.000000,2003.000000
25%,1.0,92102.000000,34.030915,-121.815412,9.000000,35.500000,0.000000,40.000000,3469.000000
50%,1.0,93552.000000,36.391777,-119.730885,29.000000,70.350000,0.000000,61.000000,4527.000000
75%,1.0,95351.000000,38.224869,-118.043237,55.000000,89.850000,1.000000,75.000000,5380.500000
max,1.0,96161.000000,41.962127,-114.192901,72.000000,118.750000,1.000000,100.000000,6500.000000


In [5]:
df.describe(exclude = np.number)

,CustomerID,Country,State,City,Lat Long,Gender,Senior Citizen,Partner,Dependents,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Total Charges,Churn Label,Churn Reason
count,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043.0,7043,1869
unique,7043,1,1,1129,1652,2,2,2,2,2,3,3,3,3,3,3,3,3,3,2,4,6531.0,2,20
top,3668-QPYBK,United States,California,Los Angeles,"33.964131, -118.272783",Male,No,No,No,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,20.2,No,Attitude of support person
freq,1,7043,7043,305,5,3555,5901,3641,5416,6361,3390,3096,3498,3088,3095,3473,2810,2785,3875,4171,2365,11.0,5174,192


#### Observações

| Coluna | Erro / Suspeita | Ação |
|:------:|:---------------:|:----:|
|`Total Charges`| Está como tipo ***object***| Converter para ***float***|
|`Senior Citizen`,`Partner`,`Dependents`, `Phone Service`, `Paperless Billing`| Está como tipo ***str*** | Converter para ***Int (0 = No, 1 = Yes)***|
|`Multiple Lines`, `Online Security`, `Online Backup`, `Device Protection`, `Tech Support`, `Streaming TV`, `Streaming Movies` | Pode ser representado como campo ***booleano*** |Converter para ***Int (0 = No or 'Não aplicável', 1 = Yes)***|
|`count`, `country`, `state`| Possui somente um único valor de ocorrência | Remover esses campos|
|`Churn Label`, `Churn Value`| Apresentam a mesma informação | Excluir a coluna `Churn Label` e renomear a `Churn Value` para `Target`|
|`Churn Score`,`Churn Reason`| Data leakage | Remover Colunas|
|`Zip Code`, `Lat Long`| Representam a mesma informação que as colunas `Latitude` `Longitude`| Remover Colunas|
|`CustomerID`| É um ID único. Não tem poder preditivo e pode causar overfitting| Remover Coluna|


* A princípio vendo o resultado do `df.info()` somente a coluna `Churn Reason` apresenta missing values
* A coluna `CustomerID` não possui ID repetido, o que indica que nenhum cliente se repete nesse dataset

#### Conversão dos tipos de dado

In [6]:
# Transformando str em bool

cols_to_bool = [
    'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Senior Citizen',
    'Online Security', 'Online Backup', 'Device Protection', 
    'Tech Support', 'Streaming TV', 'Streaming Movies', 'Paperless Billing'
]
change = {
    'Yes': 1,
    'No': 0,
    'No phone service': 0,
    'No internet service': 0,
}

df[cols_to_bool] = df[cols_to_bool].replace(change).astype(np.int64)

# Total charge possui campo com ' ', por isso está como object
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors = 'coerce')

print(df.dtypes)
df.head()

CustomerID               str
Count                  int64
Country                  str
State                    str
City                     str
Zip Code               int64
Lat Long                 str
Latitude             float64
Longitude            float64
Gender                   str
Senior Citizen         int64
Partner                int64
Dependents             int64
Tenure Months          int64
Phone Service          int64
Multiple Lines         int64
Internet Service         str
Online Security        int64
Online Backup          int64
Device Protection      int64
Tech Support           int64
Streaming TV           int64
Streaming Movies       int64
Contract                 str
Paperless Billing      int64
Payment Method           str
Monthly Charges      float64
Total Charges        float64
Churn Label              str
Churn Value            int64
Churn Score            int64
CLTV                   int64
Churn Reason             str
dtype: object


,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,0,0,0,2,1,0,DSL,1,1,0,0,0,0,Month-to-month,1,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,0,0,1,2,1,0,Fiber optic,0,0,0,0,0,0,Month-to-month,1,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,0,0,1,8,1,1,Fiber optic,0,0,1,0,1,1,Month-to-month,1,Electronic check,99.65,820.50,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,0,1,1,28,1,1,Fiber optic,0,0,1,1,1,1,Month-to-month,1,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,0,0,1,49,1,1,Fiber optic,0,1,1,0,1,1,Month-to-month,1,Bank transfer (automatic),103.70,5036.30,Yes,1,89,5340,Competitor had better devices


#### Análise de Missing Values

In [7]:
missing_values = pd.DataFrame({
    'Coluna': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2)
})

missing_values = (missing_values[missing_values['Missing_Count'] > 0]
                  .sort_values(by = 'Missing_Percentage', ascending = False)
                  )

if len(missing_values) > 0:
    print(missing_values)
else:
    print("Não missing value detectado!")

                      Coluna  Missing_Count  Missing_Percentage
Churn Reason    Churn Reason           5174               73.46
Total Charges  Total Charges             11                0.16


#### Estudando missing values na coluna `Total Charges`

In [8]:
df[df['Total Charges'].isnull()]

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
2234,4472-LVYGI,1,United States,California,San Bernardino,92408,"34.084909, -117.258107",34.084909,-117.258107,Female,0,1,0,0,0,0,DSL,1,0,1,1,1,0,Two year,1,Bank transfer (automatic),52.55,NaN,No,0,36,2578,NaN
2438,3115-CZMZD,1,United States,California,Independence,93526,"36.869584, -118.189241",36.869584,-118.189241,Male,0,0,0,0,1,0,No,0,0,0,0,0,0,Two year,0,Mailed check,20.25,NaN,No,0,68,5504,NaN
2568,5709-LVOEQ,1,United States,California,San Mateo,94401,"37.590421, -122.306467",37.590421,-122.306467,Female,0,1,0,0,1,0,DSL,1,1,1,0,1,1,Two year,0,Mailed check,80.85,NaN,No,0,45,2048,NaN
2667,4367-NUYAO,1,United States,California,Cupertino,95014,"37.306612, -122.080621",37.306612,-122.080621,Male,0,1,1,0,1,1,No,0,0,0,0,0,0,Two year,0,Mailed check,25.75,NaN,No,0,48,4950,NaN
2856,1371-DWPAZ,1,United States,California,Redcrest,95569,"40.363446, -123.835041",40.363446,-123.835041,Female,0,1,0,0,0,0,DSL,1,1,1,1,1,0,Two year,0,Credit card (automatic),56.05,NaN,No,0,30,4740,NaN
4331,7644-OMVMY,1,United States,California,Los Angeles,90029,"34.089953, -118.294824",34.089953,-118.294824,Male,0,1,1,0,1,0,No,0,0,0,0,0,0,Two year,0,Mailed check,19.85,NaN,No,0,53,2019,NaN
4687,3213-VVOLG,1,United States,California,Sun City,92585,"33.739412, -117.173334",33.739412,-117.173334,Male,0,1,1,0,1,1,No,0,0,0,0,0,0,Two year,0,Mailed check,25.35,NaN,No,0,49,2299,NaN
5104,2520-SGTTA,1,United States,California,Ben Lomond,95005,"37.078873, -122.090386",37.078873,-122.090386,Female,0,1,1,0,1,0,No,0,0,0,0,0,0,Two year,0,Mailed check,20.00,NaN,No,0,27,3763,NaN
5719,2923-ARZLG,1,United States,California,La Verne,91750,"34.144703, -117.770299",34.144703,-117.770299,Male,0,1,1,0,1,0,No,0,0,0,0,0,0,One year,1,Mailed check,19.70,NaN,No,0,69,4890,NaN
6772,4075-WKNIU,1,United States,California,Bell,90201,"33.970343, -118.171368",33.970343,-118.171368,Female,0,1,1,0,1,1,DSL,0,1,1,1,1,0,Two year,0,Mailed check,73.35,NaN,No,0,44,2342,NaN


#### Observação:

* Nota-se que os missing values da coluna `Total Charges` são ocorrências devido ao fato do cliente não ter realizado nenhum pagamento pelo fato de não ter concluído nenhum 1 mês conforme pode ser visto na coluna `Tenure Months`
    * Para esses casos o valor preenchido será $0$, representando que não foi feito nenhuma cobrança.

#### Realizando Data Cleansing

In [9]:
# Preenchendo missing values
# Clientes com 0 meses de casa recebem 0 de cobrança total
df['Total Charges'] = df['Total Charges'].fillna(0)

# Renomeando coluna Churn value p/ target
df.rename(columns = {'Churn Value': 'Target'}, inplace = True)

# Removendo colunas que não serão utilizadas
cols_to_remove = [
    'CustomerID', 'Count', 'Country', 'State',
    'Zip Code', 'Lat Long',
    'Churn Score', 'Churn Reason', 'Churn Label'
]

df.drop(columns = cols_to_remove, inplace = True)

print(f"Qtde missing values em Total Charge: {df['Total Charges'].isnull().sum()}")

print(f"Nova dimensionalidade da tabela: {df.shape}")
print("*="*30)
df.head()

Qtde missing values em Total Charge: 0
Nova dimensionalidade da tabela: (7043, 24)
*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=


,City,Latitude,Longitude,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Target,CLTV
0,Los Angeles,33.964131,-118.272783,Male,0,0,0,2,1,0,DSL,1,1,0,0,0,0,Month-to-month,1,Mailed check,53.85,108.15,1,3239
1,Los Angeles,34.059281,-118.307420,Female,0,0,1,2,1,0,Fiber optic,0,0,0,0,0,0,Month-to-month,1,Electronic check,70.70,151.65,1,2701
2,Los Angeles,34.048013,-118.293953,Female,0,0,1,8,1,1,Fiber optic,0,0,1,0,1,1,Month-to-month,1,Electronic check,99.65,820.50,1,5372
3,Los Angeles,34.062125,-118.315709,Female,0,1,1,28,1,1,Fiber optic,0,0,1,1,1,1,Month-to-month,1,Electronic check,104.80,3046.05,1,5003
4,Los Angeles,34.039224,-118.266293,Male,0,0,1,49,1,1,Fiber optic,0,1,1,0,1,1,Month-to-month,1,Bank transfer (automatic),103.70,5036.30,1,5340


#### Realizando verificação de anomalias nas features

In [15]:
print("🔍 Iniciando Auditoria de Integridade de Dados...\n")

# 1. Verificação de Tenure vs Total Charges
anomalias_tenure_cobranca = df[(df['Tenure Months'] == 0) & (df['Total Charges'] != 0)].shape[0]
if anomalias_tenure_cobranca > 0:
    print(f"⚠️ {anomalias_tenure_cobranca} registros encontrados com Tenure 0 e Total Charges diferente de zero!")
else:
    print("✅ Nenhuma anomalia de Tenure vs Total Charges encontrada.")

# 2. Verificação de Phone Service vs Multiple Lines
anomalias_phone = df[(df['Phone Service'] == 0) & (df['Multiple Lines'] == 1)].shape[0]
if anomalias_phone > 0:
    print(f"⚠️ {anomalias_phone} registros com Multiple Lines sem ter Phone Service!")
else:
    print("✅ Integridade de Telefonia validada.")

# 3. Verificação de Internet Service vs Serviços Adicionais
servicos_internet = ['Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies']
anomalias_internet = df[(df['Internet Service'] == 0) & (df[servicos_internet].sum(axis=1) > 0)].shape[0]
if anomalias_internet > 0:
    print(f"⚠️ {anomalias_internet} registros com serviços de Internet ativos em planos sem Internet!")
else:
    print("✅ Integridade de Serviços de Internet validada.")

# 4. Verificação de CLTV Negativo
anomalias_cltv = df[df['CLTV'] < 0].shape[0]
if anomalias_cltv > 0:
    print(f"⚠️ {anomalias_cltv} registros com CLTV negativo detectados!")
else:
    print("✅ Todos os valores de CLTV são válidos (positivos).")

# 5. Validação de Categorias (Valores Permitidos)
# Definindo os domínios aceitos
categorias_validas = {
    'Contract': ["Month-to-month", "One year", "Two year"],
    'Internet Service': ["No", "DSL", "Fiber optic", "Cable"],
    'Payment Method': ["Electronic check", "Mailed check", "Bank transfer (automatic)", "Credit card (automatic)"]
}

for coluna, valores_aceitos in categorias_validas.items():
    # Verifica se existe algum valor no DF que NÃO está na lista de aceitos
    invalidos = df[~df[coluna].isin(valores_aceitos)][coluna].unique()
    if len(invalidos) > 0:
        print(f"⚠️ Valores inesperados na coluna {coluna}: {invalidos}")
    else:
        print(f"✅ Coluna {coluna} contém apenas valores permitidos.")
        
# 6. Verificação de Linhas Duplicadas
total_duplicatas = df.duplicated().sum()
if total_duplicatas > 0:
    print(f"⚠️ {total_duplicatas} linhas duplicadas (idênticas) detectadas no dataset!")
    # Opcional: df = df.drop_duplicates()
else:
    print("✅ Nenhuma linha duplicada detectada.")

print("\n🚀 Auditoria finalizada!")

🔍 Iniciando Auditoria de Integridade de Dados...

✅ Nenhuma anomalia de Tenure vs Total Charges encontrada.
✅ Integridade de Telefonia validada.
✅ Integridade de Serviços de Internet validada.
✅ Todos os valores de CLTV são válidos (positivos).
✅ Coluna Contract contém apenas valores permitidos.
✅ Coluna Internet Service contém apenas valores permitidos.
✅ Coluna Payment Method contém apenas valores permitidos.
✅ Nenhuma linha duplicada detectada.

🚀 Auditoria finalizada!


In [64]:
# Persistindo dados limpos
path_save_csv = os.path.join('..', 'data', 'processed', 'df_telco_cleaned.csv')

df.to_csv(path_save_csv, index = False)